# Week 2 — Representations: from letters to models

**27200 Data-driven Bioengineering**

Today you build three biological representations on real data.

| | |
|---|---|
| **Part 1** | One-hot encoding — the simplest representation, and what it assumes |
| **Part 2** | The alphabet is a choice — BLOSUM as an encoding |
| **Part 3** | A family, an alignment, a profile HMM — and searching with it |
| **Part 4** | Post-translational modifications — what a 20-letter alphabet cannot say |

**Time:** about 35-45 minutes. **Backbone principles:** 3 (representation), 4 (models & abstraction),
5 (measurement defines reality).

Proteins are fetched live from UniProt, so you are working with the real annotation, not a
toy dataset. Run the cells in order.

## 0. Setup

Run this once. It takes about 30 seconds.

In [ ]:
!pip install -q pyhmmer pyfamsa biopython

In [ ]:
import requests, numpy as np, matplotlib.pyplot as plt
import pyfamsa, pyhmmer
from pyhmmer.easel import Alphabet, TextSequence, TextMSA, DigitalSequenceBlock
from pyhmmer.plan7 import Builder, Background

AA = "ACDEFGHIKLMNPQRSTVWY"          # the 20-letter alphabet
API = "https://rest.uniprot.org/uniprotkb"

def uniprot(query, fields, size=500):
    '''Run a UniProt search and return a list of dicts.'''
    r = requests.get(f"{API}/search",
                     params={"query": query, "format": "tsv", "fields": fields, "size": size},
                     timeout=90)
    r.raise_for_status()
    rows = [l.split("\t") for l in r.text.strip().split("\n")]
    return [dict(zip(rows[0], v)) for v in rows[1:]]

def entry(acc):
    '''Fetch one full UniProt entry as JSON (sequence + all annotation).'''
    r = requests.get(f"{API}/{acc}.json", timeout=90); r.raise_for_status()
    return r.json()

print("ready")

---
## Part 1 — One-hot encoding

One-hot encoding turns a sequence of letters into a matrix of numbers by giving every amino acid its
own column. Residue *i* becomes a row that is 1 in the column for its amino acid and 0 everywhere
else.

A sequence of length *L* becomes an **L × 20** matrix.

In [ ]:
def one_hot(seq, alphabet=AA):
    '''Encode a protein sequence as an (L x 20) one-hot matrix.'''
    idx = {a: i for i, a in enumerate(alphabet)}
    M = np.zeros((len(seq), len(alphabet)))
    for i, aa in enumerate(seq):
        if aa in idx:                 # anything not in the alphabet stays an all-zero row
            M[i, idx[aa]] = 1
    return M

# The N-terminal tail of human histone H3.1 - we will come back to this sequence a lot.
h3_tail = "ARTKQTARKSTGGKAPRKQLATKAARKSAPATGGVKKPHR"

X = one_hot(h3_tail)
print("sequence length :", len(h3_tail))
print("matrix shape    :", X.shape)
print("numbers stored  :", X.size)
print("non-zero        :", int(X.sum()), f"({100*X.sum()/X.size:.1f}% of the matrix)")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.imshow(one_hot(h3_tail).T, aspect="auto", cmap="Greys", interpolation="nearest")
ax.set_yticks(range(20)); ax.set_yticklabels(AA, fontsize=7)
ax.set_xticks(range(0, len(h3_tail), 5))
ax.set_xticklabels([str(i+1) for i in range(0, len(h3_tail), 5)], fontsize=8)
ax.set_xlabel("position in H3 tail"); ax.set_ylabel("amino acid")
ax.set_title("One-hot encoding of the histone H3 N-terminal tail", fontsize=10)
plt.tight_layout(); plt.show()

### Exercise 1

Two questions the picture above should make you ask.

1. A typical human protein is about 400 residues. How many numbers does its one-hot matrix hold, and
   how many of them are non-zero?
2. One-hot says every amino acid is *equally different* from every other. Check it: encode `"AIL"`
   and `"AIW"` and measure the distance between them. Then do `"AIL"` vs `"AIK"`. Are they the same?

Isoleucine and leucine are near-interchangeable in a protein; isoleucine and tryptophan are not.
Does your distance know that?

In [ ]:
# 1. Scale
L = 400
# YOUR CODE HERE: how many numbers, and how many non-zero?

# 2. Distances
def dist(a, b):
    '''Euclidean distance between the one-hot encodings of two equal-length sequences.'''
    # YOUR CODE HERE
    ...

# print("AIL vs AIW:", dist("AIL", "AIW"))
# print("AIL vs AIK:", dist("AIL", "AIK"))

---
## Part 2 — The alphabet is a choice

The distances above came out identical because one-hot encodes only *identity*. Nothing else.

A substitution matrix such as BLOSUM62 encodes something we actually know: how often one amino acid
replaces another in real alignments of related proteins. If we use a **row of BLOSUM62** as the
vector for each amino acid instead of a one-hot row, similarity becomes part of the representation
rather than something the model has to rediscover.

This is what the lecture called an **inductive bias**: a piece of prior knowledge built into the
representation itself.

In [ ]:
from Bio.Align import substitution_matrices
B = substitution_matrices.load("BLOSUM62")

def blosum_encode(seq):
    '''Encode a sequence as (L x 20) using rows of BLOSUM62.'''
    return np.array([[B[aa, b] for b in AA] for aa in seq], dtype=float)

pairs = [("I", "L", "conservative - both aliphatic"),
         ("I", "V", "conservative"),
         ("I", "W", "radical - small aliphatic vs big aromatic"),
         ("I", "K", "radical - hydrophobic vs charged"),
         ("D", "E", "conservative - both acidic")]

print(f"{'pair':7} {'one-hot':>9} {'BLOSUM':>9}   note")
for a, b, note in pairs:
    d_oh = np.linalg.norm(one_hot(a) - one_hot(b))
    d_bl = np.linalg.norm(blosum_encode(a) - blosum_encode(b))
    print(f"{a}->{b:4} {d_oh:9.2f} {d_bl:9.2f}   {note}")

One-hot gives the same number for every pair. BLOSUM separates conservative from radical
substitutions, because that information was measured from real alignments and put into the encoding
on purpose.

**Neither is "correct".** One-hot makes no claim, which is safe but uninformative. BLOSUM makes a
strong claim, which helps when it is true for your problem and hurts when it is not — a matrix built
from globular protein families is a poor prior for a disordered region or a designed sequence.

---
## Part 3 — A family, an alignment, and a profile HMM

Encoding one sequence is a representation of that sequence. To represent a *family* we need
something position-specific: at this position the family is almost always glycine, at that one it
tolerates anything.

That is what a **profile hidden Markov model** is, and it is what sits behind Pfam and InterPro
(lecture slides 31–32). We build one now, in four steps:

1. fetch a family from UniProt
2. align it
3. build the HMM
4. search a set of new sequences with it

Our family: **histone H3**, whose N-terminal tail we have been encoding. It is one of the most
conserved proteins in eukaryotes and it is where essentially all histone modifications reside.

In [ ]:
# 1. Fetch reviewed histone H3 entries across eukaryotes
fam = uniprot('(protein_name:"Histone H3") AND (reviewed:true) AND (length:[130 TO 140])',
              "accession,id,organism_name,sequence")
fam = sorted(fam, key=lambda r: r["Entry"])[:30]      # keep it small and deterministic
print(f"{len(fam)} sequences")
for r in fam[:5]:
    print(f"  {r['Entry']}  {r['Entry Name']:<12} {r['Organism'][:40]}")
print("  ...")

In [ ]:
# 2. Align just the N-terminal tail (first 45 residues)
TAIL = 45
seqs = [pyfamsa.Sequence(r["Entry Name"].encode(), r["Sequence"][:TAIL].encode()) for r in fam]
aln = pyfamsa.Aligner(guide_tree="upgma").align(seqs)

print(f"alignment: {len(list(aln))} sequences x {len(aln[0].sequence.decode())} columns\n")
for s in list(aln)[:8]:
    print(f"  {s.id.decode():<12} {s.sequence.decode()}")

In [ ]:
# 3. Build the profile HMM
abc = Alphabet.amino()

def build_hmm(aligned, name=b"H3"):
    msa = TextMSA(name=name,
                  sequences=[TextSequence(name=s.id, sequence=s.sequence.decode()) for s in aligned])
    hmm, _, _ = Builder(abc).build_msa(msa.digitize(abc), Background(abc))
    return hmm

hmm_tail = build_hmm(aln, b"H3_tail")
print("match states in the model:", hmm_tail.M)

In [ ]:
# What the model actually stores: a probability for every amino acid at every position
E = np.array(hmm_tail.match_emissions)[1:]        # (M x 20); row 0 is unused

fig, ax = plt.subplots(figsize=(11, 3.4))
im = ax.imshow(E.T, aspect="auto", cmap="viridis", interpolation="nearest")
ax.set_yticks(range(20)); ax.set_yticklabels(AA, fontsize=7)
ax.set_xlabel("match state (position in the profile)"); ax.set_ylabel("amino acid")
ax.set_title("Profile HMM emission probabilities - the family, as a representation", fontsize=10)
plt.colorbar(im, ax=ax, label="P(amino acid | position)")
plt.tight_layout(); plt.show()

print("Compare this with the one-hot picture in Part 1.")
print("One-hot: one sequence, hard 0/1.   Profile: a whole family, graded probabilities.")

In [ ]:
# 4. Search a set of sequences we did NOT train on
test = {"P68431": "Histone H3.1 (human)",
        "P84243": "Histone H3.3 (human)",
        "P61830": "Histone H3 (baker's yeast)",
        "P49450": "CENP-A (human) - a centromeric H3 variant",
        "P62805": "Histone H4 (human) - a different histone",
        "P0C0S8": "Histone H2A (human) - a different histone",
        "P02144": "Myoglobin (human) - unrelated",
        "P61626": "Lysozyme C (human) - unrelated"}

rows = uniprot(" OR ".join(f"accession:{a}" for a in test), "accession,sequence", 50)
targets = DigitalSequenceBlock(abc, [
    TextSequence(name=r["Entry"].encode(), sequence=r["Sequence"]).digitize(abc) for r in rows])

def scan(hmm, targets, labels):
    hits = list(pyhmmer.hmmsearch([hmm], targets, E=1000))[0]
    got = {(h.name.decode() if isinstance(h.name, bytes) else h.name): h.score for h in hits}
    print(f"{'accession':10} {'bit score':>10}   description")
    for acc, desc in labels.items():
        s = f"{got[acc]:.1f}" if acc in got else "no hit"
        print(f"{acc:10} {s:>10}   {desc}")
    return got

scores_tail = scan(hmm_tail, targets, test)

Read that table carefully. The interesting result is **not** detecting other histone proteins.

The three real histone H3 proteins score around 95 bits. The other histones and the two unrelated
proteins are rejected, which is what we want.

But **CENP-A is not found**, and CENP-A *is* a histone H3 variant — it replaces H3 at centromeres.
We missed a genuine family member.

### Exercise 2

Why did we miss it? We trained on the **first 45 residues only**. CENP-A's N-terminal tail is
completely divergent, even though its histone-fold core is not.

Rebuild the model on the **full-length** sequences and search again.

In [ ]:
# Align and build on the FULL sequences instead of the first 45 residues
# YOUR CODE HERE:
#   1. build pyfamsa Sequence objects from the complete r["Sequence"]
#   2. align them
#   3. build_hmm(...)
#   4. scan(...) against the same targets
#
# Then answer: what happened to CENP-A, and what happened to the H3 scores?

Keep that result:

> A profile is only as general as the region you built it from. A negative result from a database
> search is a statement about your model, not about biology.

It is also the reason **structure** search exists. Foldseek or DALI would have found CENP-A
immediately, because the fold is conserved even where the sequence is not. You will use exactly this
in the group assignment.

---
## Part 4 — Post-translational modifications: what the alphabet cannot say

Everything so far assumed a 20-letter alphabet. Real proteins are not written in 20 letters.

Histone H3 is the extreme case, which is why we chose it. Let us ask UniProt what it actually knows
about this one protein.

In [ ]:
h3 = entry("P68431")
seq = h3["sequence"]["value"]
mods = [f for f in h3["features"] if f["type"] == "Modified residue"]

print(f"human histone H3.1 ({len(seq)} residues)")
print(f"annotated modified-residue features : {len(mods)}")
print(f"distinct positions modified         : {len({f['location']['start']['value'] for f in mods})}")
print(f"distinct chemistries                : {len({f['description'].split(';')[0] for f in mods})}")

More annotated modification states than residues, on a protein of 136 amino acids.

**A coordinate trap first.** UniProt numbers the sequence from the initiator methionine. Histone
biologists number from the residue *after* it, because the Met is cleaved. So the famous **H3K4** is
**position 5** in UniProt. Every time you cross between a database and a literature convention,
check this — it is one of the most common silent errors in the field.

In [ ]:
print("UniProt position 5 :", seq[4], "  <- this is 'H3K4' in the histone literature")
print("UniProt position 10:", seq[9], "  <- this is 'H3K9'")

# group the annotations into broad chemical classes
def mod_class(desc):
    d = desc.lower()
    if "phospho"  in d: return "phospho"
    if "acetyl"   in d: return "acetyl"
    if "methyl"   in d: return "methyl"
    return "other"

CLASSES = ["phospho", "acetyl", "methyl", "other"]
print("\nby class:")
for c in CLASSES:
    n = sum(1 for f in mods if mod_class(f["description"]) == c)
    print(f"  {c:9} {n:4}")

print("\nfirst few sites (histone numbering):")
for f in mods[:8]:
    p = f["location"]["start"]["value"]
    print(f"  {seq[p-1]}{p-1:<4} {f['description'][:58]}")

Note what landed in `other`: crotonylation, lactylation, 2-hydroxyisobutyrylation,
beta-hydroxybutyrylation, benzoylation, citrullination. These are real, experimentally verified modifications. Even
"20 letters plus phospho, acetyl and methyl" is a simplification of a simplification.

### Putting modifications into the representation

The lecture gave four options. The one that works best with what we already built is to **add
channels**: keep the 20 one-hot columns and append one column per modification class. A residue is
still an S; it is now *also* flagged as phosphorylated.

The encoding goes from **L × 20** to **L × 24**.

In [ ]:
def ptm_channels(length, sites, classes=CLASSES):
    '''sites: list of (position_1based, class). Returns (L x len(classes)).'''
    P = np.zeros((length, len(classes)))
    ci = {c: i for i, c in enumerate(classes)}
    for pos, cls in sites:
        if cls in ci and 1 <= pos <= length:
            P[pos-1, ci[cls]] = 1
    return P

def encode(seq, sites=()):
    '''One-hot, optionally with PTM channels appended.'''
    return np.hstack([one_hot(seq), ptm_channels(len(seq), sites)])

print("plain one-hot     :", one_hot(h3_tail).shape[1], "columns  (L x 20)")
print("with PTM channels :", encode(h3_tail, [(5, "methyl")]).shape[1], "columns  (L x 24)")

### Distinguishing between two proteoforms

Two molecules of histone H3. Identical amino acid sequence. One carries trimethylation at K4, the
other at K9.

- **H3K4me3** marks an active gene promoter.
- **H3K9me3** marks silenced heterochromatin.

Opposite biological meaning. Now ask each representation whether it can tell them apart.

In [ ]:
# same sequence, two different modification states (UniProt numbering: K4 -> 5, K9 -> 10)
K4me3 = [(5, "methyl")]
K9me3 = [(10, "methyl")]

d_seq = np.linalg.norm(one_hot(h3_tail) - one_hot(h3_tail))
d_ptm = np.linalg.norm(encode(h3_tail, K4me3) - encode(h3_tail, K9me3))

print(f"sequence identical?                {h3_tail == h3_tail}")
print(f"distance, one-hot only          :  {d_seq:.2f}   <- indistinguishable")
print(f"distance, one-hot + PTM channels:  {d_ptm:.2f}   <- distinguishable")
print()
print("The biology differs. The 20-letter representation does not.")
print("Whatever model you put on top - a classifier, an HMM, a protein language model -")
print("cannot recover information the encoding threw away.")

### Exercise 3 — the honest encoding is a fraction, not a bit

A `1` in a PTM channel asserts that the site is modified on **100% of the molecules, in every
condition**. That is almost never what the data says. A phosphosite might be 5% occupied at rest and
60% occupied after stimulation, and occupancy is what carries the signal.

1. Rewrite the PTM channel so it holds an **occupancy between 0 and 1** instead of a flag.
2. Encode the same tail at rest (K4me3 at 0.05) and after stimulation (K4me3 at 0.60), and measure
   the distance.
3. Then think about a harder one, and write your answer as a comment: H3 has ~15 well-known
   modifiable sites in the tail. If each is independently present or absent, how many distinct
   proteoforms are there? Can a per-position channel matrix represent "K4me3 **and** K9me3 on the
   same molecule" as distinct from "K4me3 in half the molecules, K9me3 in the other half"?

In [ ]:
def ptm_channels_occupancy(length, sites, classes=CLASSES):
    '''sites: list of (position, class, occupancy in [0,1]).'''
    # YOUR CODE HERE
    ...

# rest vs stimulated
# ...

# How many proteoforms from 15 independent binary sites?
# YOUR ANSWER:

---
## What to take away

1. **One-hot is a claim, not a neutral default.** It says: order matters, all residues are equally
   different. That second claim is false, and everything downstream inherits it.
2. **BLOSUM, profiles and HMMs are ways of putting knowledge into the representation** so the model
   does not have to learn it from scratch. That is an inductive bias, and it helps exactly when it
   is true for your data.
3. **A profile only generalises as far as what you built it from.** We missed a real H3 variant
   because we trained on the tail. Negative database results describe your model, not biology.
4. **The 20-letter alphabet silently deletes the regulatory layer.** H3K4me3 and H3K9me3 have
   opposite meanings and identical sequences. Adding channels is cheap; deciding what the channel
   *means* (present? occupancy? on which molecule?) is the real modelling work.
5. **Protein language models are trained on the same 20 letters.** ESM has never seen a
   phosphoserine. Keep that in mind in Week 11 when the embeddings look like they know everything.

### Next

The group assignment takes this to a protein of your own: find its homologs by sequence, then by
structure, and see where the two disagree. Part 3 is the reason those two searches do not return the
same answer.